# 03 · Data visualization with Matplotlib and Seaborn

**Objective:** To learn how to create a variety of plots to explore and understand datasets using the Matplotlib and Seaborn libraries.

## Part 1: Concepts (90 mins)

### 1. What are Matplotlib and Seaborn?
- **Matplotlib:** The foundational plotting library in Python. It's highly customizable but can sometimes be verbose.
- **Seaborn:** Built on top of Matplotlib, Seaborn provides a high-level interface for drawing attractive and informative statistical graphics. It often requires less code for complex plots.

The names of the parts of a Matplotlib figure (Figure, Axes, ticks, spines, legend...) are shown in the [Anatomy of a figure](https://matplotlib.org/stable/gallery/showcase/anatomy.html) example of the Matplotlib gallery. The two most important ones:
* the **Figure** (`fig`) is the whole image, the "canvas";
* an **Axes** (`ax`) is one plot inside it, with its own x and y axis, title and labels. A figure can contain several Axes side by side.

**Two ways of writing Matplotlib code.** You will see both in this notebook, and everywhere online:

| | Object-oriented style | `pyplot` style |
|---|---|---|
| Create | `fig, ax = plt.subplots()` | `plt.figure()` |
| Draw | `ax.plot(x, y)`, `ax.hist(...)` | `plt.plot(x, y)`, `plt.hist(...)` |
| Title | `ax.set_title('...')` | `plt.title('...')` |
| Axis labels | `ax.set_xlabel('...')`, `ax.set_ylabel('...')` | `plt.xlabel('...')`, `plt.ylabel('...')` |
| Show | `plt.show()` | `plt.show()` |

In the `pyplot` style, every `plt.` command acts on "the current plot", the last one created. It is quick for a single plot. The object-oriented style says explicitly *which* Axes you are drawing on, which is necessary as soon as a figure has more than one plot. Note that the names change slightly: `plt.title` becomes `ax.set_title`.

**Reading the first example below:**
* `np.linspace(0, 10, 100)` creates 100 evenly spaced x values between 0 and 10, and `np.sin(x)` computes the sine of all of them (vectorisation, notebook 01).
* `plt.subplots()` creates a figure with one Axes and returns **both**, as a pair: `fig, ax = ...` unpacks the pair into two variables.
* `ax.plot(x, y)` draws a line through the points `(x[0], y[0]), (x[1], y[1]), ...`. The two arrays must have the same length.
* `plt.show()` displays the figure. (In Jupyter the figure is also shown automatically at the end of a cell, but `plt.show()` makes it explicit and is needed in scripts.)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simple line plot
x = np.linspace(0, 10, 100)
y = np.sin(x)

fig, ax = plt.subplots()  # Create a figure and an axes.
ax.plot(x, y)
ax.set_title("A Simple Sine Wave")
ax.set_xlabel("X-axis")
ax.set_ylabel("Y-axis")
plt.show()



**More than one plot in a figure.** `plt.subplots(1, 2)` creates a grid of **1 row and 2 columns** of Axes. It now returns the figure and an *array* of two Axes, which `(ax1, ax2)` unpacks: each plot command is then sent to the Axes it belongs to, `ax1.plot(...)` on the left, `ax2.plot(...)` on the right. For a 2 × 2 grid: `fig, axes = plt.subplots(2, 2)`, then `axes[0, 1]` is the plot in the first row, second column.

Try the commented-out line: `figsize=[6, 3]` sets the size of the figure in inches, and `layout="constrained"` automatically adjusts the spacing so that titles and labels do not overlap. Similarly, the commented `ax1.scatter` draws the points without connecting them.

In [ ]:
# Simple line plot
x = np.linspace(0, 10, 100)
y_sin = np.sin(x)
y_cos = np.cos(x)


#fig, (ax1,ax2) = plt.subplots(1,2, figsize=[6,3], layout="constrained") 
fig, (ax1,ax2) = plt.subplots(1,2)  # Create a figure and two axes.
#ax1.scatter(x, y_sin)
ax1.plot(x, y_sin)
ax1.set_title("A Simple Sine Wave")
ax1.set_xlabel("X-axis")
ax1.set_ylabel("Y-axis")

ax2.plot(x, y_cos)
ax2.set_title("A Simple Cosine Wave")
ax2.set_xlabel("X-axis")
ax2.set_ylabel("Y-axis")

plt.show()

**Saving a figure to a file.** `fig.savefig('name.png')` writes the figure to disk; the format is chosen from the extension (`.png`, `.pdf`, `.svg`, `.jpg`). Use PDF or SVG for papers and slides (vector formats: sharp at any zoom), PNG for the web. `dpi=` sets the resolution of PNG files, and `bbox_inches='tight'` trims the empty margins. In the `pyplot` style, the equivalent is `plt.savefig(...)`, which must be called **before** `plt.show()`: after showing, the figure is closed and an empty image would be saved.

In [ ]:
fig.savefig('sine_cosine.png', dpi=150, bbox_inches='tight')
print("Saved: open sine_cosine.png from the file browser on the left.")

### 2. Loading Our Dataset: The Iris Dataset

To make our plots interesting, we'll use the classic Iris dataset (Fisher, 1936). It contains four measurements for 150 iris flowers, 50 for each of three species. We'll load it using Scikit-learn and convert it to a pandas DataFrame.

* `load_iris()` returns an object with several attributes: `iris.data` is a NumPy array of shape `(150, 4)` with the measurements, `iris.feature_names` the names of the four columns, `iris.target` an array with the species of each flower coded as a number (`0`, `1`, `2`).
* `pd.DataFrame(data=..., columns=...)` builds a DataFrame from a NumPy array, using the list of names for the columns.
* `iris_df['species'] = iris.target` adds a new column.
* `series.map(dictionary)` replaces each value using the dictionary: `0` becomes `'setosa'`, and so on. Readable names are much better than codes for plot labels and legends.
* The last line of a cell, written without `print`, is displayed as a formatted table: that is why `iris_df.head()` looks nicer than `print(iris_df.head())`.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
iris = load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df['species'] = iris.target

# Map target numbers to species names for better labels
species_map = {0: 'setosa', 1: 'versicolor', 2: 'virginica'}
iris_df['species_name'] = iris_df['species'].map(species_map)

print("Iris Dataset ready for visualization:")
iris_df.head()

#### What are we measuring?

Each row of the dataset is one flower, described by four lengths: the **sepals** are the three large outer parts that droop downwards (the ones with the yellow and white veins in the picture below), the **petals** are the three smaller, upright purple parts in between. (The pale, narrow parts in the very centre are the style branches, part of the reproductive organs: they are not measured.)

A picture is also a good excuse to see how to **open an image file**: `plt.imread` reads it into a NumPy array of shape `(height, width, 3)`: one value for red, green and blue for every pixel. `ax.imshow` displays such an array.

In [ ]:
img = plt.imread('iris_versicolor.jpg')
print("Image array:", img.shape, img.dtype)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.imshow(img)
ax.axis('off')  # no ticks or frame around a photo
ax.set_title('Iris versicolor')
plt.show()

# Photo: Danielle Langlois, CC BY-SA 3.0, via Wikimedia Commons
# https://commons.wikimedia.org/wiki/File:Iris_versicolor_3.jpg

### 3. `matplotlib.pyplot`: Your Plotting Toolbox
Let's create some fundamental plots with Matplotlib. The examples in this section use the `pyplot` style:

* `plt.figure(figsize=(width, height))` starts a new, empty figure. The size is in inches; the default is `(6.4, 4.8)`.
* Then each `plt.` command adds something to it: the data, the title, the axis labels, the legend.
* `plt.show()` displays it.

#### Histogram
Shows the distribution of a single numerical variable: the range of values is divided into intervals (*bins*), and the height of each bar is **how many values fall in that interval**.

`plt.hist(values, bins=15, edgecolor='black')`:
* `values`: a column (Series), array or list of numbers;
* `bins`: the number of intervals. Too few hide the shape of the distribution; too many make it noisy. Try changing it!
* `edgecolor='black'`: draws a black outline around the bars, so that neighbouring bars can be told apart. Purely cosmetic.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(iris_df['sepal length (cm)'], bins=15, edgecolor='black')
plt.title('Distribution of Sepal Length')
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Frequency')
plt.show()

#### Scatter Plot
Shows the relationship between two numerical variables: each row of the table becomes one point, with one column on the x axis and another on the y axis. Coloring by species makes it much more insightful.

`plt.scatter(x_values, y_values, label=...)` draws one set of points. To get one colour per species, the code calls it **once per species**:

* `for name, group in iris_df.groupby('species_name'):` loops over the groups of notebook 02. At each step, `name` is the species name (e.g. `'setosa'`) and `group` is a smaller DataFrame with only the rows of that species.
* Each new `plt.scatter` call automatically gets the next colour of the default palette.
* `label=name` gives a name to each set of points, and `plt.legend()` collects these labels into the legend box. Without `label=`, the legend would be empty.

In [ ]:
plt.figure(figsize=(10, 7))

# Plot each species as a different color
for name, group in iris_df.groupby('species_name'):
    plt.scatter(group['sepal length (cm)'], group['sepal width (cm)'], label=name)

plt.title('Sepal Length vs. Sepal Width')
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Sepal Width (cm)')
plt.legend()
plt.show()

#### Bar Chart
Compares a numerical value across different categories: one bar per category, whose height is the value.

* First, `groupby` computes one number per species (notebook 02). The result is a Series: its **index** holds the species names, its **values** the averages.
* `plt.bar(categories, heights, color=[...])` draws the bars: `mean_petal_length.index` gives the positions/labels on the x axis, `mean_petal_length.values` the heights. The optional `color` list gives one colour per bar.

A bar chart shows only one number per category (here the mean) and hides how spread out the values are. The box plot in the next section shows more.

In [ ]:
mean_petal_length = iris_df.groupby('species_name')['petal length (cm)'].mean()

plt.figure(figsize=(8, 6))
plt.bar(mean_petal_length.index, mean_petal_length.values, color=['blue', 'orange', 'green'])
plt.title('Average Petal Length by Species')
plt.ylabel('Average Petal Length (cm)')
plt.show()

### 4. `seaborn`: High-Level Statistical Plotting
Seaborn makes many common statistical plots much easier to create. It is imported as `sns`, and almost all its functions share the same syntax:

```python
sns.boxplot(data=iris_df, x='species_name', y='petal width (cm)', hue='species_name')
```
* `data=`: the DataFrame;
* `x=`, `y=`: the **names** of the columns (as strings) to put on each axis;
* `hue=` (optional): the name of a column used to **colour** the data, with a legend added automatically.

Compare with the scatter plot above: in Matplotlib we had to loop over the species, pick the columns, and set the labels ourselves. Seaborn does the grouping, the colours, the legend and the axis labels (taken from the column names) in one call. Lab 3.2 asks you to redo that scatter plot with `sns.scatterplot`.

Seaborn draws on ordinary Matplotlib Axes, so you can still customise the result with `plt.title`, `plt.xlabel`, `plt.figure(figsize=...)`... as before.

#### Box Plot
Shows the distribution of a numerical variable across categories, highlighting the median, quartiles, and outliers. One box per category. How to read a box:

* the **line inside the box** is the **median** (50% of the values are below it);
* the **box** goes from the first quartile (25%) to the third quartile (75%): it contains the middle half of the data. Its height is the *interquartile range* (IQR);
* the **whiskers** extend to the furthest data points that are within 1.5 × IQR of the box;
* **points beyond the whiskers** are drawn individually: potential *outliers*.

These are the same numbers that `describe()` gives in notebook 02 (`25%`, `50%`, `75%`), drawn as a picture.

In [ ]:
plt.figure(figsize=(10, 7))
sns.boxplot(x='species_name', y='petal width (cm)', data=iris_df)
plt.title('Distribution of Petal Width by Species')
plt.show()

#### Pair Plot
This is one of Seaborn's most powerful plots. It creates scatter plots for all pairs of numerical features and histograms for each feature, all in one grid. The `hue` parameter colors the data by a categorical variable.

* With 4 numerical columns, the grid is 4 × 4. The plot in row *i*, column *j* is the scatter plot of feature *j* (x) against feature *i* (y). The grid is symmetric: the plots above the diagonal are the mirror images of those below it.
* On the **diagonal** a feature would be plotted against itself, so the distribution of that feature is shown instead (with `hue`, as smooth curves, one per species).
* `pairplot` uses **every numeric column**: we drop the numeric `species` code (`0, 1, 2`), otherwise it would be plotted as if it were a measurement.
* `height=2.5` is the size of each small panel, in inches. `pairplot` creates its own figure, so `plt.figure(figsize=...)` is not needed here.
* `plt.suptitle` sets a title for the whole figure (not a single panel); `y=1.02` places it slightly above the top edge, so that it does not overlap the panels.

In [ ]:
# We drop the original 'species' column as 'hue' needs the categorical names
sns.pairplot(iris_df.drop('species', axis=1), hue='species_name', height=2.5)
plt.suptitle('Pairwise Relationships in the Iris Dataset', y=1.02) # Adjust title position
plt.show()

## Part 2: Exercises & Debugging (90 mins)

### Lab 3.1: Histogram of Petal Length
* **Task:** Create a histogram using Matplotlib to show the distribution of `petal length (cm)` from the `iris_df`.

In [ ]:
# Your code here


### Lab 3.2: Scatter Plot of Petal Dimensions
* **Task:** The relationship between petal length and petal width is a very strong indicator of the species. Create a scatter plot of `petal length (cm)` vs. `petal width (cm)`. Color the points by species.

In [ ]:
# Your code here


### Lab 3.3: Box Plot of Sepal Length
* **Task:** Use Seaborn to create a box plot that compares the distributions of `sepal length (cm)` for each of the three species.

In [ ]:
# Your code here


## Part 3: Back to our own data

The same tools applied to the company and sales datasets of notebook 02. The next cell reloads them, with the same steps as in notebook 02:

* `company_df`: the employees table merged with the departments table, so that each employee has a `department_name`;
* `df_sales`: the sales data, with the missing quantities filled with the median and the region names cleaned up (`fillna`, then `.str.strip().str.title()`).

`sns.set_theme(style="whitegrid")` changes the look of all the following plots: Seaborn's default colours and fonts, with a light grid in the background.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Company data (created by database_setup.py)
df_emp = pd.read_csv('employees.csv')
df_dept = pd.read_csv('departments.csv')
company_df = pd.merge(df_emp, df_dept, on='department_id')

# Sales data, with the basic cleanup of notebook 02
df_sales = pd.read_csv('messy_sales_data.csv')
df_sales['Quantity'] = df_sales['Quantity'].fillna(df_sales['Quantity'].median())
df_sales['Region'] = df_sales['Region'].fillna('Unknown').str.strip().str.title()

sns.set_theme(style="whitegrid")

### Lab 3.4: Salary vs. Tenure
* **Task:** Create a scatter plot of `salary` against `tenure_years` for the company data, coloring the points by `department_name`.

In [ ]:
# Your code here


### Lab 3.5: Sales Quantity Distribution
* **Task:** Create a histogram to show the distribution of sales quantities in the cleaned `df_sales`.
* **Hint:** quantities are integers: look at the `discrete` argument of `sns.histplot`.

In [ ]:
# Your code here


### Lab 3.6: Box Plot of Prices by Region
* **Task:** Using the cleaned sales data, create a box plot to compare the distribution of prices across regions. How many orders are behind each box? Is a box plot a good choice here?

In [ ]:
# Your code here


### Lab 3.7: Correlation Heatmap
* **Task:** A heatmap is a good way to visualize correlations. Calculate the correlation matrix of the four numerical columns of `iris_df` and display it as a heatmap.
* **Hint:** Use the `.corr()` method on the DataFrame.

**Background.** The (Pearson) correlation between two columns is a number between −1 and +1: close to +1 if one tends to grow when the other grows, close to −1 if one decreases when the other grows, around 0 if there is no linear relation. `df.corr()` computes it for every pair of columns, giving a square table. `sns.heatmap(table)` draws such a table as a grid of coloured squares. Useful arguments: `annot=True` writes the number in each square, `cmap=` chooses the colour map (`'coolwarm'` is a good choice for values with a sign), `vmin=` and `vmax=` fix the range of the colour scale.

In [ ]:
# Your code here


### Debugging Focus: Choosing the Right Plot
- **The Problem:** You have data, but you don't know which plot to use to answer your question.
- **The Solution:** Think about what you want to show:
    - **Distribution of one variable?** Use a `histogram` or `boxplot`.
    - **Relationship between two numerical variables?** Use a `scatterplot`.
    - **Comparing a number across categories?** Use a `barplot`.
    - **Distribution across categories?** Use a `boxplot` or `violinplot`.
    - **Exploring all relationships at once?** A `pairplot` is your best friend.

### Debugging Focus: Overlapping Labels
- **The Problem:** When you have many categories on an axis, the labels can overlap and become unreadable.
- **Example:** A bar chart with 20 different departments.
- **How to fix:**
  1.  **Increase Figure Size:** `plt.figure(figsize=(width, height))` gives the plot more room.
  2.  **Rotate Labels:** `plt.xticks(rotation=45)` tilts the x-axis labels.
  3.  **Swap Axes:** If it makes sense, make it a horizontal bar chart so labels have more vertical space (`sns.barplot(x=value, y=category)`).